# SemEval 2024: MGT Detection Pipeline (Dual-GPU)

**Hardware**: T4 x2 (15GB + 15GB = 30GB VRAM total).

This notebook runs the full pipeline:
1. Feature Extraction (GPT-2 medium + large) with DataParallel
2. Classifier Training (Transformer Encoder) with plots
3. Prediction + Report-quality plots & analysis
4. Export pickle files for fusion with team member
5. Push everything to Hugging Face

In [ ]:
import os, shutil

# Auto-copy project files from /input to /working
found = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(('.py', '.jsonl', '.txt', '.json')):
            shutil.copy(os.path.join(root, f), os.path.join("/kaggle/working", f))
            found.append(f)
print(f"Imported {len(found)} files: {', '.join(found)}")

In [ ]:
!pip install -r requirements.txt huggingface_hub --quiet
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

### Step 1: Hugging Face Login (Kaggle Secrets)
Uses Kaggle Secrets to securely authenticate with Hugging Face without pasting tokens.
Make sure you have added a secret named **`HF_TOKEN`** in your Kaggle Notebook's Add-ons -> Secrets menu.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully logged in to Hugging Face via Kaggle Secrets!")
except Exception as e:
    print("Error reading HF_TOKEN:", e)
    print("Please ensure you created a Kaggle Secret named HF_TOKEN!")

### Step 2: Feature Extraction (Dual-GPU)
Uses DataParallel to split batches of 64 texts across both T4 GPUs.

In [ ]:
!python extract_features.py

### Step 2.5: Scrub NaN values
Fixes any float16 precision artifacts before training.

In [ ]:
import torch, numpy as np, os
for split in ["train", "dev", "test"]:
    path = f"features/{split}_features.npy"
    if os.path.exists(path):
        feats = torch.tensor(np.load(path))
        n_nan = torch.isnan(feats).sum().item()
        feats = torch.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
        np.save(path, feats.numpy())
        print(f"{split}: scrubbed {n_nan} NaN values")

### Step 3: Classifier Training
Trains the Transformer Encoder classifier and generates **training curves** + **LR schedule** plots.

In [ ]:
!python train_classifier.py

### Step 4: Predictions + Report Plots
Runs inference and generates all report-quality plots:
- Confusion Matrix
- Classification Report (image + text)
- ROC Curve (with AUC)
- Precision-Recall Curve (with AP)
- Prediction Confidence Distribution
- Feature Distributions (violin plots)
- Feature Correlation Heatmap
- t-SNE Visualization

In [ ]:
!python predict.py --split test --output final_predictions.json

### Step 4.5: Display All Generated Plots
Preview everything we just generated.

In [ ]:
import os
from IPython.display import display, Image, Markdown

plot_dir = "plots"
if os.path.isdir(plot_dir):
    plots = sorted([f for f in os.listdir(plot_dir) if f.endswith('.png')])
    print(f"Found {len(plots)} plots:\n")
    for p in plots:
        display(Markdown(f"### {p.replace('_', ' ').replace('.png', '').title()}"))
        display(Image(filename=os.path.join(plot_dir, p), width=700))
        print()
else:
    print("No plots directory found.")

In [ ]:
# Display metrics summaries
import json

for f in ["results/evaluation_metrics.json", "results/training_summary.json"]:
    if os.path.exists(f):
        print(f"\n{'='*60}")
        print(f"  {f}")
        print(f"{'='*60}")
        with open(f) as fh:
            print(json.dumps(json.load(fh), indent=2))

cr = "results/classification_report.txt"
if os.path.exists(cr):
    print(f"\n{'='*60}")
    print(f"  Classification Report")
    print(f"{'='*60}")
    with open(cr) as fh:
        print(fh.read())

### Step 5: Export Pickle Files for Team Fusion
Exports two types of pickle files per split:
1. **Raw 12-D features** with id + label (for feature-level fusion)
2. **256-D embeddings** from trained classifier with id + label + predictions (for embedding-level fusion)

In [ ]:
!python export_pickles.py --output-dir pickle_exports

In [ ]:
# Verify exported pickles
import pickle, os

pkl_dir = "pickle_exports"
if os.path.isdir(pkl_dir):
    print(f"{'File':<50s} {'Size':>10s}  {'Rows':>8s}  {'Cols':>6s}")
    print("-" * 80)
    for f in sorted(os.listdir(pkl_dir)):
        if f.endswith('.pkl'):
            path = os.path.join(pkl_dir, f)
            size = os.path.getsize(path) / (1024*1024)
            with open(path, 'rb') as fh:
                data = pickle.load(fh)
            rows = len(data)
            sample_id = next(iter(data.keys()))
            cols = len(data[sample_id]['vector'])
            print(f"{f:<50s} {size:>8.1f} MB  {rows:>8,}  {cols:>6}")
else:
    print("No pickle exports found.")

### Step 6: Push Everything to Hugging Face
Uploads checkpoints, plots, metrics, predictions, and **pickle files** to your HuggingFace repo.

In [ ]:
from huggingface_hub import HfApi
import os, glob, time

api = HfApi()
REPO_NAME = "MGT-Detection-SemEval2024"

try:
    username = api.whoami()['name']
    repo_id = f"{username}/{REPO_NAME}"
    print(f"Creating repository: {repo_id}")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

    # Collect all files to upload
    upload_files = []

    for f in glob.glob("checkpoints/*"): upload_files.append(f)
    for f in glob.glob("plots/*.png"): upload_files.append(f)
    for f in glob.glob("results/*"): upload_files.append(f)
    for f in glob.glob("pickle_exports/*.pkl"): upload_files.append(f)
    if os.path.exists("final_predictions.json"): upload_files.append("final_predictions.json")

    print(f"\nUploading {len(upload_files)} files...")
    for file in upload_files:
        if os.path.exists(file):
            size_mb = os.path.getsize(file) / (1024*1024)
            
            # Retry loop to easily handle Kaggle Network Timeouts
            for attempt in range(5):
                try:
                    # Set environment variable to force longer timeout tolerance
                    os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
                    
                    api.upload_file(
                        path_or_fileobj=file,
                        path_in_repo=file,
                        repo_id=repo_id,
                        repo_type="model"
                    )
                    print(f"  ✓ {file} ({size_mb:.1f} MB) uploaded!")
                    break
                except Exception as e:
                    print(f"  [Attempt {attempt+1}/5] Timeout uploading {file}. Retrying in 5s...")
                    time.sleep(5)
            else:
                print(f"  X Failed to upload {file} after 5 attempts.")

    print(f"\n{'='*60}")
    print(f"  All files uploaded!")
    print(f"  https://huggingface.co/{repo_id}")
    print(f"{'='*60}")

except Exception as e:
    print(f"Upload failed: {e}")
    import traceback
    traceback.print_exc()